#  Veri Setinin Yapısal Dönüşümü ve Özellik Mühendisliği (Feature Engineering)

Bu çalışmanın temel amacı, UCI Forest Fires veri setindeki ham değişkenlerin, makine öğrenmesi algoritmaları tarafından daha verimli modellenebilmesi için istatistiksel ve etki-tabanlı dönüşümlere tabi tutulmasıdır. Süreç kapsamında, model genelleme yeteneğini düşüren ve özellik sızıntısına (feature leakage) yol açabilecek değişkenler ayıklanmış; meteorolojik, zamansal ve mekansal özellikleri matematiksel düzlemde temsil eden yeni endeksler türetilmiştir. Elde edilen yapı, "Özellik Kayması" (Covariate Shift) riskini minimize edecek şekilde kurgulanmıştır.

In [1]:
import pandas as pd
import numpy as np
import os

import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

print("Veri Boyutu:", df.shape)

df.head(10)
df.info()
df.describe(include='all')

Veri Boyutu: (517, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       517 non-null    int64  
 1   Y       517 non-null    int64  
 2   month   517 non-null    object 
 3   day     517 non-null    object 
 4   FFMC    517 non-null    float64
 5   DMC     517 non-null    float64
 6   DC      517 non-null    float64
 7   ISI     517 non-null    float64
 8   temp    517 non-null    float64
 9   RH      517 non-null    int64  
 10  wind    517 non-null    float64
 11  rain    517 non-null    float64
 12  area    517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
count,517.000000,517.000000,517,517,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000
unique,NaN,NaN,12,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,aug,sun,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,184,95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,4.669246,4.299807,NaN,NaN,90.644681,110.872340,547.940039,9.021663,18.889168,44.288201,4.017602,0.021663,12.847292
std,2.313778,1.229900,NaN,NaN,5.520111,64.046482,248.066192,4.559477,5.806625,16.317469,1.791653,0.295959,63.655818
min,1.000000,2.000000,NaN,NaN,18.700000,1.100000,7.900000,0.000000,2.200000,15.000000,0.400000,0.000000,0.000000
25%,3.000000,4.000000,NaN,NaN,90.200000,68.600000,437.700000,6.500000,15.500000,33.000000,2.700000,0.000000,0.000000
50%,4.000000,4.000000,NaN,NaN,91.600000,108.300000,664.200000,8.400000,19.300000,42.000000,4.000000,0.000000,0.520000
75%,7.000000,5.000000,NaN,NaN,92.900000,142.400000,713.900000,10.800000,22.800000,53.000000,4.900000,0.000000,6.570000


## 1. Temel Değişken Dönüşümleri (In-Place Transformations)
Veri setinde yer alan temel değişkenlerin dağılım problemleri giderilerek yapısal dönüşümleri gerçekleştirilmiştir:
1. **Yağış (`rain`):** Yağış miktarı, orman yangını dinamiklerinde sürekli (continuous) bir değişkenden ziyade kategorik bir eşik değeri olarak anlam ifade etmektedir. Bu sebeple 0'dan büyük yağış değerleri 1, diğerleri 0 olacak şekilde ikili (binary) formata dönüştürülmüştür.
2. **Yanan Alan (`area`):** Bağımlı değişkenin (hedef değişken) yüksek derecede sağa çarpık (right-skewed) dağılım sergilemesi ve yoğun sıfır (zero-inflated) değerleri barındırması nedeniyle, logaritmik dönüşüm `log(x + 1)` uygulanarak dağılımın varyans stabilizasyonu sağlanmıştır.

In [4]:
df['rain'] = (df['rain'] > 0).astype(int)

df['area'] = np.log1p(df['area'])

display(df[['rain', 'area']].head())

,rain,area
0,0,0.0
1,0,0.0
2,0,0.0
3,1,0.0
4,0,0.0


## 2. Zamansal Değişkenlerin Döngüsel (Cyclical) Kodlanması
Aylar ve haftanın günleri gibi zamansal değişkenlerin makine öğrenmesi algoritmalarına doğrusal (linear) bir formatta sunulması, "Aralık ve Ocak aylarının ardışık olması" gibi döngüsel (cyclical) ilişkilerin model tarafından anlaşılamamasına yol açmaktadır. Bu kısıtı aşmak üzere, gün ve ay değişkenleri trigonometrik (Sinüs ve Kosinüs) fonksiyonlar aracılığıyla iki boyutlu düzleme aktarılmış ve zamansal süreklilik matematiksel olarak ifade edilmiştir.

In [5]:
month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6, 
             'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
day_map = {'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7}

month_num = df['month'].map(month_map)
day_num = df['day'].map(day_map)

df['month_sin'] = np.sin(2 * np.pi * month_num / 12.0)
df['month_cos'] = np.cos(2 * np.pi * month_num / 12.0)

df['day_sin'] = np.sin(2 * np.pi * day_num / 7.0)
df['day_cos'] = np.cos(2 * np.pi * day_num / 7.0)

display(df[['month', 'month_sin', 'month_cos', 'day', 'day_sin', 'day_cos']].head())

,month,month_sin,month_cos,day,day_sin,day_cos
0,mar,1.000000,6.123234e-17,fri,-9.749279e-01,-0.222521
1,oct,-0.866025,5.000000e-01,tue,9.749279e-01,-0.222521
2,oct,-0.866025,5.000000e-01,sat,-7.818315e-01,0.623490
3,mar,1.000000,6.123234e-17,fri,-9.749279e-01,-0.222521
4,mar,1.000000,6.123234e-17,sun,-2.449294e-16,1.000000


## 3. Zamansal Risk Kategorizasyonu
İnsan kaynaklı (antropojenik) faktörlerin ve iklimsel risklerin orman yangınları üzerindeki etkisini modelleyebilmek amacıyla yapısal kategoriler oluşturulmuştur:
* **Hafta Sonu Etkisi (`is_weekend`):** İnsan aktivitesinin yoğunlaştığı Cuma, Cumartesi ve Pazar günleri yüksek riskli (1) olarak etiketlenmiştir.
* **Mevsimsel Zirve (`is_peak_season`):** İstatistiksel olarak yangın sıklığının ve yanan alan miktarının en yüksek olduğu Ağustos ve Eylül ayları kurak sezon (1) olarak tanımlanmıştır.

In [6]:
df['is_weekend'] = df['day'].isin(['sat', 'sun']).astype(int)

peak_months = ['mar', 'may', 'aug', 'sep', 'dec']
df['is_peak_season'] = df['month'].isin(peak_months).astype(int)

display(df[['day', 'is_weekend', 'month', 'is_peak_season']].head())

,day,is_weekend,month,is_peak_season
0,fri,0,mar,1
1,tue,0,oct,0
2,sat,1,oct,0
3,fri,0,mar,1
4,sun,1,mar,1


## 4. Mekansal (Spatial) Risk Özellikleri
Montesinho Doğa Parkı'nın coğrafi yapısına ait olan X ve Y eksenlerinin (1-9 ızgara sistemi) model tarafından mekansal bir bütünlük içinde algılanması sağlanmıştır:
* **Merkeze Uzaklık (`distance_to_center`):** Parkın geometrik merkezi olan (5,5) koordinatlarına göre hesaplanan Öklid uzaklığı ile coğrafi dağılım modellenmiştir.
* **Dinamik Mevsimsel Merkez Uzaklığı (`dynamic_seasonal_hotspot_distance`):** Mevsimsel faktörlere göre değişen yangın odaklarına (Yaz: 8,6; Kış: 6,5) göre hesaplanan uzaklık metrikleri kurgulanarak, yangın dinamiklerinin mekansal hareketliliği (spatial drift) tanımlanmıştır.

In [7]:
df['distance_to_center'] = np.sqrt((df['X'] - 5)**2 + (df['Y'] - 5)**2)

def calculate_dynamic_distance(row):
    
    if row['month'] in ['jun', 'jul', 'aug', 'sep']:
        target_x, target_y = 8, 6
    
    else:
        target_x, target_y = 6, 5
        
    return np.sqrt((row['X'] - target_x)**2 + (row['Y'] - target_y)**2)

df['dynamic_seasonal_hotspot_distance'] = df.apply(calculate_dynamic_distance, axis=1)

display(df[['X', 'Y', 'month', 'distance_to_center', 'dynamic_seasonal_hotspot_distance']].head())

,X,Y,month,distance_to_center,dynamic_seasonal_hotspot_distance
0,7,5,mar,2.000000,1.000000
1,7,4,oct,2.236068,1.414214
2,7,4,oct,2.236068,1.414214
3,8,6,mar,3.162278,2.236068
4,8,6,mar,3.162278,2.236068


## 5. Meteorolojik Risk Eşiklerinin Belirlenmesi ve FWI Etkileşimleri
Orman yangınlarının yayılımında temel rol oynayan hava durumu koşulları ve Yangın Hava İndeksi (FWI) bileşenleri, literatürdeki tehlike sınırları baz alınarak ikili değişkenlere (binary) ve etkileşim çarpanlarına dönüştürülmüştür:
* **Rüzgar Şiddeti (`moderate_wind_danger`):** Rüzgar hızının 3.5 km/s ile 6.0 km/s arasında olması orta/yüksek şiddetli tehlike olarak iki aşamalı şekilde modellenmiştir.
* **Sıcak ve Kuru Koşullar (`hot_and_dry`):** Sıcaklığın 20°C üzerinde ve bağıl nemin (RH) %40 veya altında olduğu durumlar kritik kuraklık sınırı olarak tanımlanmıştır.
* **Çifte Kuraklık Krizi (`double_drought`):** Yüzey kuraklığının (FFMC ≥ 88.0) ve derin katman kuraklığının (DC ≥ 500.0) eş zamanlı olarak görüldüğü yapısal kriz durumları kodlanmıştır.
* **Etkileşim Çarpanı (`ISI_x_DC`):** Yangının yayılma potansiyelini (ISI) ve zemindeki yakıt yükü kuraklığını (DC) bir araya getiren bir sürekli değişken üretilmiştir.

In [8]:
df['moderate_wind_danger'] = ((df['wind'] >= 3.5) & (df['wind'] <= 6.0)).astype(int)

df['hot_and_dry'] = ((df['temp'] >= 20.0) & (df['RH'] <= 40)).astype(int)

df['double_drought'] = ((df['FFMC'] >= 88.0) & (df['DC'] >= 500.0)).astype(int)

df['ISI_x_DC'] = df['ISI'] * df['DC']

display(df[['moderate_wind_danger', 'hot_and_dry', 'double_drought', 'ISI_x_DC']].head())

,moderate_wind_danger,hot_and_dry,double_drought,ISI_x_DC
0,0,0,0,480.93
1,0,0,1,4482.97
2,0,0,1,4602.23
3,1,0,0,697.50
4,0,0,0,981.12


## 7. Doğrusal Bağımlılıkların Giderilmesi ve Dışa Aktarım
Zamanı döngüsel (Sin/Cos) fonksiyonlarla ifade etmek üzere yeni değişkenler üretildiği için, modelde Çoklu Doğrusal Bağlantı (Multicollinearity) yaratmaması adına orijinal `month` ve `day` değişkenleri veri setinden çıkarılmıştır. Dönüşüm işlemleri tamamlanan sızıntısız veri seti analitik sürece hazır hale getirilerek kaydedilmiştir.

In [9]:
df.drop(columns=['month', 'day'], inplace=True)

print(f"Yeni Veri Boyutu: {df.shape}")
display(df.info())
display(df.head())

Yeni Veri Boyutu: (517, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 23 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   X                                  517 non-null    int64  
 1   Y                                  517 non-null    int64  
 2   FFMC                               517 non-null    float64
 3   DMC                                517 non-null    float64
 4   DC                                 517 non-null    float64
 5   ISI                                517 non-null    float64
 6   temp                               517 non-null    float64
 7   RH                                 517 non-null    int64  
 8   wind                               517 non-null    float64
 9   rain                               517 non-null    int64  
 10  area                               517 non-null    float64
 11  month_sin                     

None

,X,Y,FFMC,DMC,DC,ISI,temp,RH,wind,rain,...,day_sin,day_cos,is_weekend,is_peak_season,distance_to_center,dynamic_seasonal_hotspot_distance,moderate_wind_danger,hot_and_dry,double_drought,ISI_x_DC
0,7,5,86.2,26.2,94.3,5.1,8.2,51,6.7,0,...,-9.749279e-01,-0.222521,0,1,2.000000,1.000000,0,0,0,480.93
1,7,4,90.6,35.4,669.1,6.7,18.0,33,0.9,0,...,9.749279e-01,-0.222521,0,0,2.236068,1.414214,0,0,1,4482.97
2,7,4,90.6,43.7,686.9,6.7,14.6,33,1.3,0,...,-7.818315e-01,0.623490,1,0,2.236068,1.414214,0,0,1,4602.23
3,8,6,91.7,33.3,77.5,9.0,8.3,97,4.0,1,...,-9.749279e-01,-0.222521,0,1,3.162278,2.236068,1,0,0,697.50
4,8,6,89.3,51.3,102.2,9.6,11.4,99,1.8,0,...,-2.449294e-16,1.000000,1,1,3.162278,2.236068,0,0,0,981.12


In [ ]:
save_dir = '../../data/processed'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = f'{save_dir}/processed_forestfires_v2.csv'
df.to_csv(save_path, index=False)

print(f"\n Veri '{save_path}' yoluna kaydedildi!")


Veri '../../data/processed/processed_forestfires_v2.csv' yoluna kaydedildi!
